# Question-conditioned evidence and value selection

**Question.** Does a two-stage lexical pipeline—retrieve a table row or text passage first, then rank raw values within the retrieved evidence—improve on the 3/871 dev lexical value-selection reference?

**Protocol.** Vocabulary is fitted from train questions and train raw context only. Dev is evaluated once; the official test split is not loaded. At prediction time, the pipeline receives only ID, question, table, pre_text, and post_text. Gold answers, programs, executable answers, gold evidence, and derived artifacts are accessed only after predictions for metrics and diagnostics. This is extractive value selection, not numerical reasoning.

In [1]:
from __future__ import annotations

import json
import re
import tomllib
from pathlib import Path

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

from finevidence.data import build_structured_input, load_finqa_split
from finevidence.evaluation import ComparisonStatus, compare_display_answers, parse_display_answer

PROJECT_ROOT = (
    Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
)
CONFIG_PATH = PROJECT_ROOT / "configs" / "question_conditioned_value_selection.toml"
REFERENCE_PATH = (
    PROJECT_ROOT / "data" / "processed" / "lexical_value_baseline_dev_predictions.jsonl"
)
config = tomllib.loads(CONFIG_PATH.read_text(encoding="utf-8"))
RAW_DIR = PROJECT_ROOT / config["dataset_root"]
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"

train_records = load_finqa_split(RAW_DIR / f"{config['train_split']}.json")
dev_records = load_finqa_split(RAW_DIR / f"{config['evaluation_split']}.json")
reference_rows = [json.loads(line) for line in REFERENCE_PATH.open(encoding="utf-8")]
assert len(reference_rows) == len(dev_records)
print({"train_records": len(train_records), "dev_records": len(dev_records), "test_loaded": False})

{'train_records': 6251, 'dev_records': 883, 'test_loaded': False}


## Inference-safe evidence and candidates

A table row keeps its row label, headers, and values together. A text passage remains one raw pre_text or post_text item. Only non-label table cells, numeric spans, and yes/no spans are eligible predictions; labels remain retrieval context. Candidate raw values and source keys are preserved.

In [2]:
NUMBER_SPAN = re.compile(r"(?<![\w.])[-+]?(?:\d+(?:,\d{3})*(?:\.\d+)?|\.\d+)%?(?![\w.])")
YES_NO_SPAN = re.compile(r"\b(?:yes|no)\b", re.IGNORECASE)


def build_evidence_segments(record):
    source = build_structured_input(record)
    table = source["table"]
    header = [str(cell) for cell in table[0]]
    segments = []
    for row_index, row in enumerate(table[1:], start=1):
        pairs = " | ".join(
            f"{header[column_index]}: {value}"
            for column_index, value in enumerate(row[1:], start=1)
        )
        segments.append(
            {
                "source_key": f"table_{row_index}",
                "segment_type": "table_row",
                "context": f"row label: {row[0]} | {pairs}",
                "row": row,
                "header": header,
            }
        )
    for text_index, text in enumerate(source["pre_text"] + source["post_text"]):
        segments.append(
            {
                "source_key": f"text_{text_index}",
                "segment_type": "text",
                "context": text,
            }
        )
    return segments


def candidates_from_segment(segment):
    candidates = []
    if segment["segment_type"] == "table_row":
        for column_index, raw_value in enumerate(segment["row"][1:], start=1):
            raw_value = str(raw_value)
            if raw_value.strip():
                candidates.append(
                    {
                        "prediction": raw_value,
                        "source_key": segment["source_key"],
                        "candidate_context": (
                            f"{segment['context']} | selected column: "
                            f"{segment['header'][column_index]} value: {raw_value}"
                        ),
                    }
                )
    else:
        for pattern in (NUMBER_SPAN, YES_NO_SPAN):
            for match in pattern.finditer(segment["context"]):
                candidates.append(
                    {
                        "prediction": match.group(),
                        "source_key": segment["source_key"],
                        "candidate_context": segment["context"]
                        + f" | selected value: {match.group()}",
                    }
                )
    return candidates


sample_segments = build_evidence_segments(train_records[0])
display(pd.DataFrame(sample_segments).drop(columns=["row", "header"], errors="ignore").head(4))
display(pd.DataFrame(candidates_from_segment(sample_segments[0])).head())

,source_key,segment_type,context
0,table_1,table_row,row label: fair value of forward exchange cont...
1,table_2,table_row,row label: fair value of forward exchange cont...
2,table_3,table_row,row label: fair value of forward exchange cont...
3,text_0,text,interest rate to a variable interest rate base...


,prediction,source_key,candidate_context
0,$ 6427,table_1,row label: fair value of forward exchange cont...
1,$ -23158 ( 23158 ),table_1,row label: fair value of forward exchange cont...


## Train-only fitting and dev prediction

The vectorizer learns vocabulary from train questions and train evidence segments only. On dev, first rank evidence segments using question similarity, then rank candidates drawn only from the top retrieved segments using question-plus-local-evidence similarity.

In [3]:
train_documents = []
for record in train_records:
    source = build_structured_input(record)
    train_documents.append(source["question"])
    train_documents.extend(segment["context"] for segment in build_evidence_segments(record))

vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(config["ngram_min"], config["ngram_max"]),
    max_features=config["max_features"],
)
vectorizer.fit(train_documents)
print({"train_documents": len(train_documents), "vocabulary_size": len(vectorizer.vocabulary_)})

prediction_rows = []
for record in dev_records:
    source = build_structured_input(record)
    segments = build_evidence_segments(record)
    question_vector = vectorizer.transform([source["question"]])
    segment_matrix = vectorizer.transform([segment["context"] for segment in segments])
    segment_scores = (question_vector @ segment_matrix.T).toarray().ravel()
    ranked_segment_indices = segment_scores.argsort()[::-1][: config["top_evidence_segments"]]
    retrieved = [segments[index] for index in ranked_segment_indices]
    candidates = [
        candidate for segment in retrieved for candidate in candidates_from_segment(segment)
    ]
    if candidates:
        candidate_matrix = vectorizer.transform(
            [candidate["candidate_context"] for candidate in candidates]
        )
        candidate_scores = (question_vector @ candidate_matrix.T).toarray().ravel()
        best_index = int(candidate_scores.argmax())
        best = candidates[best_index]
        prediction, candidate_key = best["prediction"], best["source_key"]
        candidate_score = float(candidate_scores[best_index])
    else:
        prediction, candidate_key, candidate_score = None, None, None

    comparison = compare_display_answers(prediction, record["qa"]["answer"])
    parsed_reference = parse_display_answer(record["qa"]["answer"])
    prediction_rows.append(
        {
            "id": record["id"],
            "question": source["question"],
            "prediction": prediction,
            "candidate_source_key": candidate_key,
            "candidate_score": candidate_score,
            "retrieved_source_keys": [segment["source_key"] for segment in retrieved],
            "reference_answer": record["qa"]["answer"],
            "reference_type": (
                "empty"
                if parsed_reference.kind == "empty"
                else "numeric_percent"
                if parsed_reference.kind == "numeric" and parsed_reference.is_percent
                else parsed_reference.kind
            ),
            "comparison_status": comparison.status.value,
            "comparison_reason": comparison.reason,
            "metric_eligible": comparison.is_metric_eligible,
        }
    )

prediction_frame = pd.DataFrame(prediction_rows)
assert len(prediction_frame) == len(dev_records)

{'train_documents': 191354, 'vocabulary_size': 50000}


## Fixed-comparator results and reference comparison

Only after the prediction frame is fixed are reference answers used for the established display-answer metric. The existing lexical baseline is compared on the same dev records and fixed comparator.

In [4]:
eligible = prediction_frame[prediction_frame["metric_eligible"]]
correct = eligible["comparison_status"].eq(ComparisonStatus.CORRECT.value)
metrics = pd.DataFrame(
    {
        "dev_records": [len(prediction_frame)],
        "display_answer_denominator": [len(eligible)],
        "empty_reference_answers": [(~prediction_frame["metric_eligible"]).sum()],
        "correct": [correct.sum()],
    }
)
metrics["display_answer_accuracy"] = metrics["correct"] / metrics["display_answer_denominator"]
display(metrics)

per_type = (
    eligible.assign(correct=correct)
    .groupby("reference_type")
    .agg(denominator=("correct", "size"), correct=("correct", "sum"))
    .assign(accuracy=lambda frame: frame["correct"] / frame["denominator"])
)
display(per_type)

reference_frame = pd.DataFrame(reference_rows)
reference_eligible = reference_frame[reference_frame["metric_eligible"]]
comparison = pd.DataFrame(
    {
        "system": ["lexical-value-selection reference", "question-conditioned selection"],
        "correct": [
            reference_eligible["comparison_status"].eq(ComparisonStatus.CORRECT.value).sum(),
            correct.sum(),
        ],
        "denominator": [len(reference_eligible), len(eligible)],
    }
)
comparison["accuracy"] = comparison["correct"] / comparison["denominator"]
display(comparison)

,dev_records,display_answer_denominator,empty_reference_answers,correct,display_answer_accuracy
0,883,871,12,3,0.003444


,denominator,correct,accuracy
reference_type,,,
numeric,356,3,0.008427
numeric_percent,492,0,0.000000
text,23,0,0.000000


,system,correct,denominator,accuracy
0,lexical-value-selection reference,3,871,0.003444
1,question-conditioned selection,3,871,0.003444


## Post-prediction evidence and reasoning diagnostics

The next cell reads dev gold evidence and program annotations only after predictions exist. It is a limited diagnostic: it can show whether a gold key was retrieved and whether the published program has multiple steps or a table operation. It cannot establish a causal error label such as retrieval or reasoning failure.

In [5]:
POST_HOC_OPERATION = re.compile(r"([a-z_]+)\(")
post_hoc_rows = []
for prediction, record in zip(prediction_rows, dev_records, strict=True):
    operations = POST_HOC_OPERATION.findall(record["qa"]["program"])
    gold_keys = set(record["qa"]["gold_inds"])
    retrieved_keys = set(prediction["retrieved_source_keys"])
    post_hoc_rows.append(
        {
            "id": record["id"],
            "gold_evidence_retrieved": bool(gold_keys & retrieved_keys),
            "multiple_gold_sources": len(gold_keys) > 1,
            "multiple_program_steps": len(operations) > 1,
            "table_operation_present": any(
                operation.startswith("table_") for operation in operations
            ),
            "temporal_cue": bool(
                re.search(
                    r"\b(?:from|to|between|year|quarter|ended|versus|compared)\b",
                    prediction["question"],
                    re.I,
                )
            ),
            "unit_or_conversion_cue": bool(
                re.search(
                    r"\b(?:percent|percentage|million|billion|thousand|per)\b",
                    prediction["question"],
                    re.I,
                )
            ),
            "comparison_status": prediction["comparison_status"],
        }
    )
post_hoc_frame = pd.DataFrame(post_hoc_rows)
display(
    post_hoc_frame.drop(columns=["id", "comparison_status"]).sum().rename("dev_records").to_frame()
)
incorrect_diagnostics = post_hoc_frame[
    prediction_frame["comparison_status"].ne(ComparisonStatus.CORRECT.value).to_numpy()
]
display(
    incorrect_diagnostics.drop(columns=["id", "comparison_status"])
    .sum()
    .rename("incorrect_or_unavailable")
    .to_frame()
)
display(
    prediction_frame.loc[
        :,
        [
            "id",
            "question",
            "prediction",
            "reference_answer",
            "candidate_source_key",
            "comparison_status",
        ],
    ].head(8)
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
output_path = OUTPUT_DIR / "question_conditioned_value_selection_dev_predictions.jsonl"
with output_path.open("w", encoding="utf-8") as handle:
    for row in prediction_rows:
        handle.write(json.dumps(row, ensure_ascii=False) + "\n")
print(output_path.relative_to(PROJECT_ROOT))

,dev_records
gold_evidence_retrieved,715
multiple_gold_sources,467
multiple_program_steps,360
table_operation_present,35
temporal_cue,517
unit_or_conversion_cue,398


,incorrect_or_unavailable
gold_evidence_retrieved,712
multiple_gold_sources,467
multiple_program_steps,360
table_operation_present,32
temporal_cue,517
unit_or_conversion_cue,398


,id,question,prediction,reference_answer,candidate_source_key,comparison_status
0,V/2008/page_17.pdf-1,what is the average payment volume per transac...,86,127.40,table_3,incorrect
1,C/2017/page_328.pdf-1,what was the percentage cumulative total retur...,500,93.5%,text_2,incorrect
2,DVN/2007/page_58.pdf-2,what percentage of the total oil and gas mmboe...,60,24.69%,table_3,incorrect
3,ETR/2011/page_341.pdf-3,in 2010 what was the net change in net revenue...,2010,18.6,text_2,incorrect
4,ETR/2004/page_213.pdf-2,what are the deferred fuel cost revisions as a...,59.1,60.3%,table_2,incorrect
5,PM/2017/page_38.pdf-1,what was the change in millions of operating i...,6.4% ( 6.4 % ),688,table_3,incorrect
6,JPM/2008/page_85.pdf-3,what was jpmorgan chase & co's common equity t...,2008,10.94%,text_24,incorrect
7,PNC/2013/page_207.pdf-1,"at the end of 2014 , the notional value of der...",$ 152,3044%,table_1,incorrect


data/processed/question_conditioned_value_selection_dev_predictions.jsonl


## Conclusion

This experiment separates evidence ranking from value selection but remains extractive. It cannot calculate differences, ratios, averages, or multi-step compositions; it also cannot safely convert units or resolve reporting periods. The executed comparison above determines whether it improved over 3/871. The post-hoc flags identify examples whose gold annotations involve multiple steps, multiple evidence sources, table operations, temporal language, or unit cues; they are scope diagnostics, not causal failure labels.

Recommendation: use the observed evidence-retrieval diagnostic together with answer accuracy to decide the next M3 experiment. If gold evidence is often absent from retrieved segments, improve retrieval; if it is present but wrong values are selected, improve value selection; if correct evidence routinely requires calculations, introduce an explicitly tested numerical-reasoning component.